# Определение перевернутого текста (0 / 180 градусов)

Весь код решения в одном ноутбуке, по порядку. Если запустить все ячейки подряд, в конце получается `submission.csv`.

**Содержание**
0. Описание подхода (текстом, следующая ячейка)
1. Импорты, флаг `TRAIN`, пути
2. Настройки обучения
3. Препроцессинг картинки (один для обучения и для предикта)
4. Данные для обучения: шрифты, тексты, реальные кропы, синтетика
5. Датасет с поворотом на 180 на лету
6. Модель TinyCNN
7. Метрики и обучение
8. Экспорт в onnx и замер скорости
9. Предикт, проверка готовой модели на валидации и `submission.csv`

**Как запускать**
* из корня репозитория (рядом с ноутбуком должны лежать `data/` и `weights/`)
* данные из задания положить в `data/sample_submission.csv` и `data/test/images/*.png`
* зависимости: `pip install -r requirements.txt`
* флаг `TRAIN` в первой ячейке с кодом:
  * `TRAIN = False` (по умолчанию): берем готовые веса из `weights/model.onnx`, сабмит получается за пару минут
  * `TRAIN = True`: все с нуля. Качаются шрифты, тексты и реальные кропы (около 1.5 ГБ), генерится синтетика,
    учится сетка. На cpu с 4 ядрами это часа два, из них обучение около 80 минут. Что уже скачано или сгенерено, второй раз не делается.

## 0. Описание подхода

**Задача и метрика.** По картинке бокса с текстом нужно предсказать `p_180`, вероятность того что текст повернут на 180 градусов.
Метрика 1 - Brier, где Brier это среднее от (p - y)^2. Константа 0.5 дает 0.75, это нижняя планка.
Brier сильно наказывает уверенные ошибки: если уверенно ответить неправильно, штраф 1, а если честно ответить 0.5, штраф всего 0.25.
Поэтому важно не только угадывать, но и чтобы вероятности были честными.

**Главная идея: разметка бесплатная.** Обучающих данных не дали, но для этой задачи они и не нужны в обычном смысле.
Любая картинка с текстом, которая стоит правильно, после поворота на 180 становится примером класса 1.
Значит надо собрать много правильно стоящих кропов текста, похожих на тест, а метку делать самому
случайным поворотом прямо во время обучения. Одна и та же картинка попадает то в класс 0, то в класс 1,
так что сетке не за что зацепиться кроме самого текста.

**Что в тесте** (подробно в `notebooks/01_eda.ipynb`). 20 000 кропов с фоток объявлений: вывески, упаковки, скриншоты,
номера машин, логотипы, иногда рукописный текст. В основном русский, но латиницы тоже много. Треть картинок ниже 32 пикселей
по высоте, много размытых. Вертикального текста почти нет, поэтому поворот на 90 не рассматриваю.

**Данные для обучения** (примеры в `notebooks/02_data_preview.ipynb`):
1. *Синтетика, 300 тысяч картинок.* Тексты из открытого датасета товаров ozon (названия, описания, характеристики)
   и шаблоны (телефоны, цены, номера машин, адреса, фразы из объявлений). 204 семейства шрифтов с кириллицей из Google Fonts.
   Текст рисуется на случайном фоне, иногда с обводкой, тенью и кусками соседних строк, потом немного поворачивается,
   бокс режется как у неидеального детектора, картинка уменьшается и портится (блюр, шум, jpeg).
2. *Реальные кропы, 66 тысяч,* из открытых OCR датасетов: COCO-Text, ICDAR 2015, TextOCR (английские слова с фоток)
   и русский рукописный текст с номерами машин. В них настоящий блюр, шум и освещение, которые синтетикой сложно повторить.

В каждом батче 30% реальных кропов и 70% синтетики.

**Препроцессинг.** Серая картинка, высота 32, ширина по пропорциям, нормировка по самой картинке, по центру холста 32 x 256.
Если картинка длиннее, она сжимается по горизонтали. Это одна функция и для обучения, и для предикта.

**Модель.** Маленькая сверточная сеть TinyCNN (273 тысячи параметров), по устройству похожа на сверточную часть CRNN из OCR:
несколько блоков свертка + batchnorm + relu, по высоте картинка сжимается с 32 до 1, по ширине результат усредняется,
в конце линейный слой выдает один логит. Сеть специально маленькая: в проде она будет работать на каждом боксе.
В onnx на одном ядре cpu это около 1 мс на бокс, файл 1 МБ.

**Обучение.** BCE с label smoothing 0.02, AdamW, OneCycle, 10 эпох по 150 тысяч картинок, около 80 минут на cpu с 4 ядрами.

**Валидация.** Меток для теста нет, поэтому проверяю на данных где метки известны:
* *synth_val:* 10 тысяч синтетических картинок, где и шрифты (27 семейств), и тексты не пересекаются с трейном.
  Это проверка что сетка учит форму букв, а не запоминает конкретные шрифты
* *real val:* 25 тысяч реальных кропов из тестовых частей COCO-Text, ICDAR 2015, TextOCR и отложенные 20% русских рукописных и номеров
* поворот на валидации фиксированный (сид), так что цифры между запусками сравнимы. Лучшую эпоху выбираю по среднему из двух скоров

Минус: реальная валидация в основном из коротких английских слов, а в тесте больше русского и длинных строк.
По валидации видно, что длинные строки сильно проще коротких, так что на тесте скор скорее всего не хуже чем на real val.

**Результаты (1 - Brier на валидации).**

| модель | synth_val | real val | параметров | мс на бокс (1 ядро cpu) |
|---|---|---|---|---|
| константа 0.5 | 0.750 | 0.750 | | |
| TinyCNN w0.5 (сеть в 2 раза уже) | 0.960 | 0.882 | 69 тыс. | 0.4 |
| **TinyCNN w1.0 (финал)** | **0.972** | **0.902** | 273 тыс. | 1.0 |

По источникам у финальной модели: COCO-Text 0.906, ICDAR 2015 0.924, TextOCR 0.892, русский рукописный 0.967, номера машин 0.994.
Вероятности получились нормально откалиброванными и без отдельной калибровки (график в `notebooks/03_results.ipynb`).
Самое сложное для модели: короткие боксы из 1-3 символов и стилизованные логотипы.

**Что можно сделать дальше:** сеть пошире (переход с w0.5 на w1.0 дал заметный прирост), очень длинные строки не сжимать,
а резать на куски и усреднять, усреднять предсказание с повернутой картинкой, найти больше реальных русских кропов.

## 1. Импорты, флаг TRAIN и пути

Все библиотеки есть в `requirements.txt`. Самое важное тут это флаг `TRAIN`:
`False` значит взять готовые веса и сразу сделать сабмит, `True` значит пройти весь путь с нуля.

In [1]:
# стандартная библиотека
import html                      # чистка html тегов в описаниях товаров
import json
import os
import random
import re
import sys
import time
import urllib.request            # скачивание шрифтов и датасетов
import zlib                      # хеш строки, по нему делим тексты на train и val
from multiprocessing import get_context   # генерация синтетики в несколько процессов

# сторонние библиотеки
import cv2                       # ресайз, блюр, jpeg и прочая работа с картинками
import numpy as np
import onnxruntime as ort        # быстрый инференс на cpu
import pandas as pd
import pyarrow.parquet as pq     # датасеты с huggingface лежат в parquet
import torch
import torch.nn as nn
import torch.nn.functional as F
from fontTools.ttLib import TTFont             # чтобы узнать какие буквы есть в шрифте
from PIL import Image, ImageDraw, ImageFont    # рисование текста для синтетики
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

# True: все с нуля (скачать данные, сгенерить синтетику, обучить сетку, часа два на cpu)
# False: взять готовые веса из weights/model.onnx и сразу сделать сабмит
TRAIN = False

ROOT = os.path.abspath('.')  # ноутбук запускаем из корня репозитория
DATA_DIR = os.path.join(ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')          # скачанные parquet файлы
FONTS_DIR = os.path.join(DATA_DIR, 'fonts')      # шрифты
TEXTS_DIR = os.path.join(DATA_DIR, 'texts')      # корпус строк для синтетики
REAL_DIR = os.path.join(DATA_DIR, 'real')        # реальные кропы
SYNTH_DIR = os.path.join(DATA_DIR, 'synth')      # синтетика
WEIGHTS = os.path.join(ROOT, 'weights', 'model.onnx')   # финальная модель
SUBMISSION = os.path.join(ROOT, 'submission.csv')

assert os.path.exists(os.path.join(DATA_DIR, 'sample_submission.csv')), 'положи данные из задания в data/ (см. README)'
print('TRAIN =', TRAIN)

TRAIN = False


## 2. Настройки обучения

Все параметры в одном словаре (то же самое лежит в `config.yaml` для запуска скриптами). В коде ниже магических чисел почти нет,
все берется отсюда.

In [2]:
CFG = {
    'seed': 42,                        # один сид для torch, numpy и random
    'input': {
        'width': 256,                  # ширина входа сетки, высота всегда 32
    },
    'data': {
        # имена пачек картинок в data/synth/ и data/real/ (как они готовятся, см. раздел 4)
        'synth_train': ['train'],
        'synth_val': ['val'],
        'real_train': ['coco_train', 'ic15_train', 'textocr_train', 'hw_ru_train', 'plates_ru_train'],
        'real_val': ['coco_val', 'ic15_val', 'textocr_val', 'hw_ru_val', 'plates_ru_val'],
        'real_share': 0.3,             # доля реальных кропов в батче, остальное синтетика
    },
    'model': {
        'width_mult': 1.0,             # множитель числа каналов (0.5 значит сеть в 2 раза уже)
    },
    'train': {
        'epochs': 10,
        'samples_per_epoch': 150000,   # сколько картинок в одной эпохе (семплируем с возвращением)
        'batch_size': 256,
        'lr': 0.003,                   # максимальный lr для OneCycle
        'weight_decay': 0.0001,
        'label_smoothing': 0.02,       # часть примеров в принципе неразличима ("ОНО", "88"), не даем сетке быть слишком уверенной
        'threads': 4,                  # потоки torch на cpu
        # воркеры даталоадера из ноутбука нормально работают только на линуксе (через fork)
        'workers': 2 if sys.platform.startswith('linux') else 0,
    },
}

## 3. Препроцессинг картинки

Самое важное место во всем решении: препроцессинг ОДИН для синтетики, для реальных кропов и для теста.
Если в обучении и в предикте картинки готовить по разному, получится тихий баг, который потом долго ищешь.

Как готовим картинку:
1. переводим в серый: цвет для ориентации текста почти не нужен, а данных в 3 раза меньше
2. ресайзим до высоты 32 с сохранением пропорций. Ширину ограничиваем `MAX_STORE_W` (512), если картинка длиннее, сжимаем по горизонтали
3. уже перед сеткой: сжимаем до ширины модели (256) если надо, нормируем и ставим по центру пустого холста 32 x 256

Почему по центру: тогда пустые поля слева и справа одинаковые у обоих классов, и сетка не может угадать ответ по тому,
с какой стороны паддинг.

In [3]:
HEIGHT = 32          # высота входа сетки
MAX_STORE_W = 512    # до такой ширины храним заготовленные картинки (длиннее сжимаем)



def seed_everything(seed=42):
    """Фиксируем все генераторы случайных чисел, чтобы результат повторялся от запуска к запуску."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def read_image(path):
    # cv2.imread не любит кириллицу в путях, поэтому через imdecode
    data = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'не смог прочитать {path}')
    return img  # BGR


def to_gray(img):
    # цвет для ориентации текста почти не нужен, а серая картинка в 3 раза меньше
    if img.ndim == 2:
        return img
    # у png бывает альфа канал, он не нужен
    if img.shape[2] == 4:
        img = img[:, :, :3]
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [4]:
def resize_to_height(gray, height=HEIGHT, max_w=MAX_STORE_W):
    """Серую картинку любого размера приводим к высоте height, ширина пропорционально (но не больше max_w)."""
    h, w = gray.shape[:2]
    new_w = int(round(w * height / h))
    # совсем узкие растягиваем хотя бы до 4 пикселей, слишком длинные сжимаем до max_w
    new_w = max(4, min(new_w, max_w))
    # уменьшаем через INTER_AREA (меньше артефактов), увеличиваем линейной
    interp = cv2.INTER_AREA if h > height else cv2.INTER_LINEAR
    return cv2.resize(gray, (new_w, height), interpolation=interp)


def prepare_image(img, height=HEIGHT, max_w=MAX_STORE_W):
    """
    Главная функция препроцессинга: на входе BGR или серая картинка любого размера, на выходе uint8 массив (height, w).
    Через нее проходит все: синтетика, реальные кропы и тест.
    """
    return resize_to_height(to_gray(img), height, max_w)

In [5]:
def to_model_input(arr, width):
    """
    Из uint8 массива (32, w) делаем float32 тензор (1, 32, width) для сетки.
    Если картинка шире чем width, то сжимаем по горизонтали. Потом нормируем по самой картинке
    (минус среднее, делим на std) и ставим по центру холста из нулей.
    """
    h, w = arr.shape
    if w > width:
        arr = cv2.resize(arr, (width, h), interpolation=cv2.INTER_AREA)
        w = width
    x = arr.astype(np.float32)
    # нормировка по самой картинке убирает разницу в яркости и контрасте между фотками
    x = (x - x.mean()) / (x.std() + 1.0)  # +1 чтобы не делить на ноль на пустых картинках
    out = np.zeros((1, h, width), dtype=np.float32)
    # ставим по центру: поля слева и справа одинаковые и никак не связаны с ориентацией текста
    left = (width - w) // 2
    out[0, :, left:left + w] = x
    return out


def rotate180(arr):
    # поворот на 180 это просто разворот по обеим осям
    return np.ascontiguousarray(arr[::-1, ::-1])

In [6]:
# Хранение заготовленных картинок.
# После препроцессинга все картинки одной высоты (32), поэтому склеиваем их по ширине в один большой массив
# и отдельно храним ширины. 300 тысяч картинок так занимают пару гигабайт и грузятся за секунды.

def save_pack(path, arrays, **extra):
    widths = np.array([a.shape[1] for a in arrays], dtype=np.int32)
    data = np.concatenate(arrays, axis=1) if arrays else np.zeros((HEIGHT, 0), np.uint8)
    np.savez(path, data=data, widths=widths, **extra)   # extra: например тексты, чтобы потом смотреть глазами


class Pack:
    """Читалка того что сохранили через save_pack: pack[i] отдает i-ю картинку (32, w)."""

    def __init__(self, path):
        z = np.load(path, allow_pickle=False)
        self.data = z['data']
        self.widths = z['widths']
        # где начинается каждая картинка в большом массиве
        self.offsets = np.concatenate([[0], np.cumsum(self.widths)])
        self.extra = {k: z[k] for k in z.files if k not in ('data', 'widths')}

    def __len__(self):
        return len(self.widths)

    def __getitem__(self, i):
        return self.data[:, self.offsets[i]:self.offsets[i + 1]]

## 4. Данные для обучения

Обучающих данных не дали, но разметка тут бесплатная (см. описание подхода): собираем много правильно стоящих кропов текста,
а метку делаем при обучении случайным поворотом. Источников два: синтетика и реальные кропы из открытых OCR датасетов.
Для синтетики нужны шрифты и тексты.

Все ячейки этого раздела при `TRAIN = False` ничего не делают.

### 4.1 Шрифты

Шрифты с кириллицей из Google Fonts. Список семейств я собрал один раз из метаданных google fonts: оставил семейства где есть
кириллица, выкинул китайские/японские/корейские (они огромные) и совсем нечитаемые декоративные (типа шрифтов из квадратиков).
Формат строки: `название;категория;начертания;train/val`.
Примерно каждое 7е семейство в каждой категории помечено как val: эти шрифты не участвуют в обучении,
на них проверяем что сетка не запомнила конкретные шрифты.

In [7]:
# 204 семейства шрифтов с кириллицей, по одному на строку
FONTS_TXT = """
Balsamiq Sans;Display;400,700,400i;train
Bellota;Display;400,700,400i;train
Bellota Text;Display;400,700,400i;train
Climate Crisis;Display;400;train
Comfortaa;Display;400,700;train
Comic Relief;Display;400,700;train
Forum;Display;400;val
Handjet;Display;400,700,900;train
Kablammo;Display;400;train
Kelly Slab;Display;400;train
Libertinus Serif Display;Display;400;train
Lobster;Display;400;train
Oi;Display;400;train
Orelega One;Display;400;val
Pixelify Sans;Display;400,700;train
Poiret One;Display;400;train
Press Start 2P;Display;400;train
Prosto One;Display;400;train
Rubik Dirt;Display;400;train
Rubik Distressed;Display;400;train
Rubik Spray Paint;Display;400;val
Rubik Wet Paint;Display;400;train
Ruslan Display;Display;400;train
Shantell Sans;Display;400,700,400i;train
Stalinist One;Display;400;train
Tektur;Display;400,700,900;train
Underdog;Display;400;train
Viaoda Libre;Display;400;val
Yeseva One;Display;400;train
Amatic SC;Handwriting;400,700;train
Bad Script;Handwriting;400;train
Caveat;Handwriting;400,700;train
Comforter;Handwriting;400;train
Comforter Brush;Handwriting;400;train
Great Vibes;Handwriting;400;train
Marck Script;Handwriting;400;val
Neucha;Handwriting;400;train
Pacifico;Handwriting;400;train
Pangolin;Handwriting;400;train
Playpen Sans;Handwriting;400,700;train
Yomogi;Handwriting;400;train
Anonymous Pro;Monospace;400,700,400i;train
Cousine;Monospace;400,700,400i;train
Fira Code;Monospace;400,700;train
Fira Mono;Monospace;400,700;train
Geist Mono;Monospace;400,700,900,400i;train
IBM Plex Mono;Monospace;400,700,400i;train
Iosevka Charon;Monospace;400,700,400i;val
Iosevka Charon Mono;Monospace;400,700,400i;train
JetBrains Mono;Monospace;400,700,400i;train
Lilex;Monospace;400,700,400i;train
Martian Mono;Monospace;400,700;train
Overpass Mono;Monospace;400,700;train
PT Mono;Monospace;400;train
Roboto Mono;Monospace;400,700,400i;val
Source Code Pro;Monospace;400,700,900,400i;train
Ubuntu Mono;Monospace;400,700,400i;train
Ubuntu Sans Mono;Monospace;400,700,400i;train
Victor Mono;Monospace;400,700,400i;train
Advent Pro;Sans Serif;400,700,900,400i;train
Akt;Sans Serif;400,700,900;train
Alegreya Sans;Sans Serif;400,700,900,400i;train
Alegreya Sans SC;Sans Serif;400,700,900,400i;train
Alumni Sans;Sans Serif;400,700,900,400i;train
Alumni Sans Collegiate One;Sans Serif;400,400i;train
Alumni Sans Pinstripe;Sans Serif;400,400i;val
Alumni Sans SC;Sans Serif;400,700,900,400i;train
Andika;Sans Serif;400,700,400i;train
Arimo;Sans Serif;400,700,400i;train
Arsenal;Sans Serif;400,700,400i;train
Arsenal SC;Sans Serif;400,700,400i;train
Carlito;Sans Serif;400,700,400i;train
Cascadia Code;Sans Serif;400,700,400i;val
Cascadia Mono;Sans Serif;400,700,400i;train
Commissioner;Sans Serif;400,700,900;train
Cuprum;Sans Serif;400,700,400i;train
Days One;Sans Serif;400;train
Didact Gothic;Sans Serif;400;train
El Messiri;Sans Serif;400,700;train
Exo 2;Sans Serif;400,700,900,400i;val
Finlandica Headline;Sans Serif;400,700,900,400i;train
Finlandica Text;Sans Serif;400,700,900,400i;train
Fira Sans;Sans Serif;400,700,900,400i;train
Fira Sans Condensed;Sans Serif;400,700,900,400i;train
Fira Sans Extra Condensed;Sans Serif;400,700,900,400i;train
Geist;Sans Serif;400,700,900,400i;train
Geologica;Sans Serif;400,700,900;val
Gidole;Sans Serif;400;train
Golos Text;Sans Serif;400,700,900;train
Google Sans;Sans Serif;400,700,400i;train
IBM Plex Sans;Sans Serif;400,700,400i;train
Inter;Sans Serif;400,700,900,400i;train
Inter Tight;Sans Serif;400,700,900,400i;train
Istok Web;Sans Serif;400,700,400i;val
Jost;Sans Serif;400,700,900,400i;train
Jura;Sans Serif;400,700;train
Libertinus Sans;Sans Serif;400,700,400i;train
Libre Franklin;Sans Serif;400,700,900,400i;train
Liter;Sans Serif;400;train
Lunasima;Sans Serif;400,700;train
Manrope;Sans Serif;400,700;val
Marmelad;Sans Serif;400;train
Moderustic;Sans Serif;400,700;train
Montserrat;Sans Serif;400,700,900,400i;train
Montserrat Alternates;Sans Serif;400,700,900,400i;train
Montserrat Underline;Sans Serif;400,700,900,400i;train
Mulish;Sans Serif;400,700,900,400i;train
Nata Sans;Sans Serif;400,700,900;val
News Cycle;Sans Serif;400,700;train
Nobile;Sans Serif;400,700,400i;train
Noto Sans;Sans Serif;400,700,900,400i;train
Noto Sans Display;Sans Serif;400,700,900,400i;train
Noto Sans Mono;Sans Serif;400,700,900;train
Nunito;Sans Serif;400,700,900,400i;train
Nunito Sans;Sans Serif;400,700,900,400i;val
Onest;Sans Serif;400,700,900;train
Open Sans;Sans Serif;400,700,400i;train
Oswald;Sans Serif;400,700;train
Overpass;Sans Serif;400,700,900,400i;train
PT Sans;Sans Serif;400,700,400i;train
PT Sans Caption;Sans Serif;400,700;train
PT Sans Narrow;Sans Serif;400,700;val
Pattaya;Sans Serif;400;train
Philosopher;Sans Serif;400,700,400i;train
Play;Sans Serif;400,700;train
Pliant;Sans Serif;400,700,900,400i;train
Raleway;Sans Serif;400,700,900,400i;train
Roboto;Sans Serif;400,700,900,400i;train
Roboto Condensed;Sans Serif;400,700,900,400i;val
Roboto Flex;Sans Serif;400,700,900;train
Rubik;Sans Serif;400,700,900,400i;train
Rubik Mono One;Sans Serif;400;train
Ruda;Sans Serif;400,700,900;train
Russo One;Sans Serif;400;train
SN Pro;Sans Serif;400,700,900,400i;train
Sansation;Sans Serif;400,700,400i;val
Scada;Sans Serif;400,700,400i;train
Science Gothic;Sans Serif;400,700,900;train
Seymour One;Sans Serif;400;train
Sofia Sans;Sans Serif;400,700,900,400i;train
Sofia Sans Condensed;Sans Serif;400,700,900,400i;train
Sofia Sans Extra Condensed;Sans Serif;400,700,900,400i;train
Sofia Sans Semi Condensed;Sans Serif;400,700,900,400i;val
Source Sans 3;Sans Serif;400,700,900,400i;train
Tenor Sans;Sans Serif;400;train
TikTok Sans;Sans Serif;400,700,900;train
Tiny5;Sans Serif;400;train
Tuffy;Sans Serif;400,700,400i;train
Ubuntu;Sans Serif;400,700,400i;train
Ubuntu Condensed;Sans Serif;400;val
Ubuntu Sans;Sans Serif;400,700,400i;train
Unbounded;Sans Serif;400,700,900;train
Wix Madefor Display;Sans Serif;400,700;train
Wix Madefor Text;Sans Serif;400,700,400i;train
Yanone Kaffeesatz;Sans Serif;400,700;train
Ysabeau;Sans Serif;400,700,900,400i;train
Ysabeau Infant;Sans Serif;400,700,900,400i;val
Ysabeau Office;Sans Serif;400,700,900,400i;train
Ysabeau SC;Sans Serif;400,700,900;train
Alegreya;Serif;400,700,900,400i;train
Alegreya SC;Serif;400,700,900,400i;train
Alice;Serif;400;train
Bitter;Serif;400,700,900,400i;train
Bona Nova;Serif;400,700,400i;train
Bona Nova SC;Serif;400,700,400i;train
Brygada 1918;Serif;400,700,400i;val
Charis SIL;Serif;400,700,400i;train
Cormorant;Serif;400,700,400i;train
Cormorant Garamond;Serif;400,700,400i;train
Cormorant Infant;Serif;400,700,400i;train
Cormorant SC;Serif;400,700;train
Cormorant Unicase;Serif;400,700;train
EB Garamond;Serif;400,700,400i;val
Gabriela;Serif;400;train
Gentium Book Plus;Serif;400,700,400i;train
Gentium Plus;Serif;400,700,400i;train
IBM Plex Serif;Serif;400,700,400i;train
Kurale;Serif;400;train
Ledger;Serif;400;train
Libertinus Serif;Serif;400,700,400i;val
Literata;Serif;400,700,900,400i;train
Lora;Serif;400,700,400i;train
Merriweather;Serif;400,700,900,400i;train
Noto Serif;Serif;400,700,900,400i;train
Noto Serif Display;Serif;400,700,900,400i;train
Old Standard TT;Serif;400,700,400i;train
Oranienbaum;Serif;400;val
PT Serif;Serif;400,700,400i;train
PT Serif Caption;Serif;400,400i;train
Piazzolla;Serif;400,700,900,400i;train
Playfair;Serif;400,700,900,400i;train
Playfair Display;Serif;400,700,900,400i;train
Playfair Display SC;Serif;400,700,900,400i;train
Podkova;Serif;400,700;val
Prata;Serif;400;train
Roboto Serif;Serif;400,700,900,400i;train
Roboto Slab;Serif;400,700,900;train
STIX Two Text;Serif;400,700,400i;train
Source Serif 4;Serif;400,700,900,400i;train
Spectral;Serif;400,700,400i;train
Spectral SC;Serif;400,700,400i;val
Tinos;Serif;400,700,400i;train
Vollkorn;Serif;400,700,900,400i;train
Vollkorn SC;Serif;400,700,900;train
"""
print(len(FONTS_TXT.strip().splitlines()), 'семейств')

204 семейств


In [8]:
# буквы которые обязательно должны быть в шрифте, иначе PIL нарисует квадратики
MUST_HAVE = ('АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюя'
             'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789')


# Как качаем: если зайти на css api google fonts без браузерного user-agent,
# он отдает css со ссылками на обычные ttf файлы. Из css достаем ссылку и качаем сам ttf.
def css_url(family, variant):
    # variant вида '400', '700', '400i'
    italic = variant.endswith('i')
    weight = variant.rstrip('i')
    fam = family.replace(' ', '+')
    if italic:
        return f'https://fonts.googleapis.com/css2?family={fam}:ital,wght@1,{weight}'
    return f'https://fonts.googleapis.com/css2?family={fam}:wght@{weight}'


def fetch(url, retries=3):
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except Exception as e:
            print('  ошибка', e, 'пробую еще раз')
            time.sleep(2 * (i + 1))
    return None


def has_all_glyphs(path):
    try:
        cmap = TTFont(path).getBestCmap()
    except Exception:
        return False
    return all(ord(ch) in cmap for ch in MUST_HAVE)


def download_fonts():
    """Качает все шрифты из FONTS_TXT в data/fonts/, уже скачанные пропускает."""
    os.makedirs(FONTS_DIR, exist_ok=True)
    lines = [l.strip() for l in FONTS_TXT.splitlines() if l.strip()]
    ok, bad = 0, 0
    for line in lines:
        family, category, variants, split = line.split(';')
        for v in variants.split(','):
            fname = f"{family.replace(' ', '_')}-{v}.ttf"
            path = os.path.join(FONTS_DIR, fname)
            if os.path.exists(path):
                ok += 1
                continue
            # сначала css, из него достаем ссылку на ttf
            css = fetch(css_url(family, v))
            if css is None:
                bad += 1
                continue
            m = re.search(r'url\((https://[^)]+\.ttf)\)', css.decode('utf-8'))
            if not m:
                print('не нашел ttf для', family, v)
                bad += 1
                continue
            data = fetch(m.group(1))
            if data is None:
                bad += 1
                continue
            with open(path, 'wb') as f:
                f.write(data)
            # проверяем что реально есть все буквы, если нет то удаляем
            if not has_all_glyphs(path):
                print('в шрифте нет нужных букв, выкидываю:', fname)
                os.remove(path)
                bad += 1
                continue
            ok += 1
    print(f'скачано {ok}, не получилось {bad}')

In [9]:
if TRAIN:
    download_fonts()  # около 500 файлов, минут 10
else:
    print('TRAIN = False, шрифты не нужны')

TRAIN = False, шрифты не нужны


### 4.2 Тексты для синтетики

Что рисовать на синтетических картинках. Нужны тексты, похожие на то что пишут на фотках объявлений: названия товаров, бренды,
размеры, цвета, "в наличии" и тд. Для этого подошел открытый датасет товаров ozon с huggingface (`evgmaslov/ozon_ecup`):
берем из него только текстовые колонки (название, описание, категории, характеристики).

На выходе два файла: `ru.txt` (строки с кириллицей, около миллиона) и `latin.txt` (только латиница: бренды, модели, английский текст).
Телефоны, цены, номера машин и тд генерим шаблонами прямо в генераторе синтетики.

In [10]:
OZON_URL = 'https://huggingface.co/datasets/evgmaslov/ozon_ecup/resolve/main/cleaned/cleaned-00000-of-00002.parquet'
OZON_FILE = os.path.join(RAW_DIR, 'ozon_cleaned_0.parquet')

# оставляем только нормальные символы, всякие эмодзи и редкие значки выкидываем
ALLOWED = re.compile('^[0-9A-Za-z\u0410-\u044f\u0401\u0451\\s.,:;!?()\\[\\]"\'\u00ab\u00bb%\u2116+\\-\u2013\u2014/&*#@_=\u20bd$\u20ac\u00b0\u00d7]+$')

def download_ozon():
    os.makedirs(RAW_DIR, exist_ok=True)
    if os.path.exists(OZON_FILE):
        return
    print('качаю', OZON_URL)
    urllib.request.urlretrieve(OZON_URL, OZON_FILE)


def clean(s):
    # убираем html теги и лишние пробелы
    s = re.sub(r'<[^>]+>', ' ', s)
    s = html.unescape(s)
    s = s.replace(' ', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def chunks(sentence, max_words=8):
    # длинные предложения режем на куски, на картинках обычно короткие строки
    words = sentence.split()
    i = 0
    while i < len(words):
        n = random.randint(2, max_words)
        yield ' '.join(words[i:i + n])
        i += n


def build_text_corpus():
    """Собирает data/texts/ru.txt и data/texts/latin.txt из датасета товаров ozon."""
    if all(os.path.exists(os.path.join(TEXTS_DIR, f)) for f in ('ru.txt', 'latin.txt')):
        print('тексты уже есть, пропускаю')
        return
    random.seed(0)  # от сида зависит как длинные предложения режутся на куски
    download_ozon()
    os.makedirs(TEXTS_DIR, exist_ok=True)
    cols = ['name', 'description', 'categories', 'characteristic_attributes_mapping']
    df = pq.read_table(OZON_FILE, columns=cols).to_pandas()
    # товары для взрослых убираем, там текст специфический
    df = df[~df.categories.fillna('').str.contains('для взрослых')]
    print('товаров:', len(df))

    # собираем все строки в множество, чтобы не было повторов
    lines = set()
    df = df.fillna('')  # в описаниях бывают пропуски (nan)
    for name, desc, cats, chars in df.itertuples(index=False):
        if name:
            name = clean(name)
            lines.add(name)
            # название часто через запятую: "Рюкзак на молнии, 2 кармана, цвет синий"
            for part in name.split(','):
                lines.add(part.strip())
        # описания режем на предложения, а длинные предложения еще на куски по 2-8 слов
        if desc:
            for sent in re.split(r'(?<=[.!?;])\s+', clean(desc)):
                for ch in chunks(sent):
                    lines.add(ch)
        # названия категорий тоже бывают на картинках
        if cats:
            for c in json.loads(cats).values():
                if c != 'EPG':
                    lines.add(c)
        # характеристики: и само значение ("синий"), и пара "Цвет: синий"
        if chars:
            for k, vals in json.loads(chars).items():
                for v in vals[:3]:
                    v = clean(str(v))
                    lines.add(v)
                    lines.add(f'{k}: {v}')

    # выкидываем слишком длинные строки и строки с редкими символами (эмодзи и тд)
    lines = [l for l in lines if 1 <= len(l) <= 80 and ALLOWED.match(l)]
    # делим на строки с кириллицей и чисто латинские. Сортируем, чтобы результат не зависел от порядка в множестве
    ru = sorted(l for l in lines if re.search('[\u0410-\u044f\u0401\u0451]', l))
    latin = sorted(l for l in lines if not re.search('[\u0410-\u044f\u0401\u0451]', l) and re.search('[A-Za-z]', l))
    # из русских строк еще вытаскиваем латинские слова (бренды типа Samsung, модели и тд)
    extra = set()
    for l in ru:
        for tok in re.findall(r'[A-Za-z][A-Za-z0-9\-+.]{1,20}', l):
            extra.add(tok)
    latin = sorted(set(latin) | extra)

    with open(os.path.join(TEXTS_DIR, 'ru.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(ru) + '\n')
    with open(os.path.join(TEXTS_DIR, 'latin.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(latin) + '\n')
    print('ru строк:', len(ru), 'latin строк:', len(latin))

In [11]:
if TRAIN:
    build_text_corpus()  # скачивание около 300 МБ и полминуты на обработку
else:
    print('TRAIN = False, тексты не нужны')

TRAIN = False, тексты не нужны


### 4.3 Реальные кропы

Любой кроп из OCR датасета стоит правильно, так что это тоже бесплатные размеченные данные.
Жаль что в основном на английском, зато тут настоящий блюр, шум, перспектива и освещение, чего в синтетике нормально не сделать.

Что берем (все с huggingface):
* COCO-Text v2, ICDAR 2015, TextOCR: слова с фоток, английский. Трейн части идут в обучение, тест части в валидацию
* Foximaz/russian_ocr_small: русский рукописный текст и номера машин, делим сами 80/20 на трейн и валидацию

Вертикальный текст и одиночные буквы выкидываем (в тесте такого почти нет). Все картинки сразу прогоняем через общий
препроцессинг и сохраняем пачками в `data/real/*.npz`.

In [12]:
HF = 'https://huggingface.co/datasets'
# имя источника: (ссылка на parquet, что с ним делать)
# train / val - целиком в обучение или в валидацию, split - делим сами 80/20
SOURCES = {
    'coco_train': (f'{HF}/Bekhouche/COCO-Text_V2_STR/resolve/main/data/train-00000-of-00001.parquet', 'train'),
    'coco_val': (f'{HF}/Bekhouche/COCO-Text_V2_STR/resolve/main/data/valid-00000-of-00001.parquet', 'val'),
    'ic15_train': (f'{HF}/MiXaiLL76/ICDAR2015_OCR/resolve/main/data/train-00000-of-00001.parquet', 'train'),
    'ic15_val': (f'{HF}/MiXaiLL76/ICDAR2015_OCR/resolve/main/data/test-00000-of-00001.parquet', 'val'),
    'textocr_train': (f'{HF}/MiXaiLL76/TextOCR_OCR/resolve/main/data/train-00000-of-00005.parquet', 'train'),
    'textocr_val': (f'{HF}/MiXaiLL76/TextOCR_OCR/resolve/main/data/test-00000-of-00001.parquet', 'val'),
    'hw_ru': (f'{HF}/Foximaz/russian_ocr_small/resolve/main/handwriting/test-00000.parquet', 'split'),
    'plates_ru': (f'{HF}/Foximaz/russian_ocr_small/resolve/main/car_plate/test-00000.parquet', 'split'),
}


def download_parquet(name, url):
    path = os.path.join(RAW_DIR, name + '.parquet')
    if not os.path.exists(path):
        print('качаю', url)
        urllib.request.urlretrieve(url, path)
    return path


def good_crop(img):
    h, w = img.shape[:2]
    if h < 8 or w < 8:
        return False
    # вертикальный текст и одиночные буквы выкидываем, в тесте такого почти нет.
    # плюс в coco-text часть слов повернута на 90, они как раз узкие и высокие
    return w >= h


def process(path):
    table = pq.read_table(path)
    images = table.column('image').to_pylist()
    texts = table.column('text').to_pylist()
    arrs, keep_texts = [], []
    for im, text in tqdm(zip(images, texts), total=len(texts), desc=os.path.basename(path)):
        if not text or not text.strip():
            continue
        # картинки внутри parquet лежат как байты png/jpg
        img = cv2.imdecode(np.frombuffer(im['bytes'], np.uint8), cv2.IMREAD_COLOR)
        if img is None or not good_crop(img):
            continue
        arrs.append(prepare_image(img))
        keep_texts.append(text.strip())
    return arrs, keep_texts


def prepare_real_crops():
    """Качает parquet файлы, режет, фильтрует и сохраняет реальные кропы в data/real/*.npz."""
    need = CFG['data']['real_train'] + CFG['data']['real_val']
    if all(os.path.exists(os.path.join(REAL_DIR, n + '.npz')) for n in need):
        print('реальные кропы уже есть, пропускаю')
        return
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(REAL_DIR, exist_ok=True)
    # источники с ролью split делим сами 80/20, сид фиксированный
    rng = np.random.RandomState(0)
    for name, (url, role) in SOURCES.items():
        path = download_parquet(name, url)
        arrs, texts = process(path)
        if role == 'split':
            idx = rng.permutation(len(arrs))
            cut = int(len(idx) * 0.8)
            parts = {name + '_train': idx[:cut], name + '_val': idx[cut:]}
        else:
            parts = {name: np.arange(len(arrs))}
        for part, ids in parts.items():
            save_pack(os.path.join(REAL_DIR, part + '.npz'), [arrs[i] for i in ids],
                      texts=np.array([texts[i] for i in ids]))
            print(f'{part}: {len(ids)} картинок')

In [13]:
if TRAIN:
    prepare_real_crops()  # около 1.1 ГБ скачивания, обработка около минуты
else:
    print('TRAIN = False, реальные кропы не нужны')

TRAIN = False, реальные кропы не нужны


### 4.4 Синтетика

Это основная часть данных (300 тысяч картинок). Каждый пример делается так:
1. берем строку: кусок из корпуса ozon, шаблон (телефон, цена, номер машины, адрес, артикул) или фразу из объявлений
2. берем случайный шрифт с кириллицей и рисуем текст на каком нибудь фоне
   (иногда с обводкой, тенью, плашкой, подчеркиванием, соседними строками сверху и снизу)
3. немного поворачиваем или искажаем перспективу и вырезаем бокс вокруг строки, как это делал бы неидеальный детектор
4. портим качество: уменьшаем до маленького разрешения, блюр, шум, jpeg
5. прогоняем через общий препроцессинг из раздела 3

Важно: картинка на выходе всегда стоит правильно, поворот и метку делаем уже при обучении.
Все "асимметричные" штуки (тень вниз, куски соседних строк) делаются ДО поворота, как и в реальности.
А поля вокруг текста сверху и снизу случайные с одинаковым распределением, чтобы сетка не могла угадать ответ по отступам.

На cpu рисовать картинки на лету во время обучения медленно, поэтому генерим их заранее и сохраняем.

Сначала константы: с какой частотой берем шрифты разных категорий и фразы для шаблонов.

In [14]:
# как часто берем шрифт из каждой категории (рукописных шрифтов мало, но в тесте такое бывает)
CATEGORY_P = {'Sans Serif': 0.45, 'Serif': 0.14, 'Display': 0.22, 'Monospace': 0.05, 'Handwriting': 0.14}

# типичные фразы из объявлений, их часто пишут на картинках
AD_PHRASES = [
    'Продам', 'Продается', 'Продаю', 'Срочно', 'Торг', 'Торг уместен', 'В наличии', 'Под заказ', 'Новый', 'Новое',
    'Б/у', 'Как новый', 'Отличное состояние', 'Хорошее состояние', 'Доставка', 'Доставка по всей России',
    'Бесплатная доставка', 'Самовывоз', 'Гарантия', 'Гарантия 1 год', 'Звоните', 'Пишите', 'Пишите в WhatsApp',
    'Звоните в любое время', 'Аренда', 'Сдается', 'Сдам', 'Сниму', 'Куплю', 'Обмен', 'Скидка', 'Скидки', 'Акция',
    'Распродажа', 'Оригинал', 'Опт и розница', 'Без посредников', 'Собственник', 'Кредит', 'Рассрочка',
    'Рассрочка без банка', 'Установка', 'Ремонт', 'Ремонт под ключ', 'Выезд мастера', 'Недорого', 'Дешево',
    'Цена договорная', 'Вопросы в лс', 'Подробнее по телефону', 'Все вопросы по телефону', 'Фото реальные',
    'Выгодно', 'Хит продаж', 'Новинка', 'Лучшая цена', 'Работаем без выходных', 'Ежедневно', 'Круглосуточно',
    'SALE', 'NEW', 'Made in Russia', 'Made in China', 'Made in Japan', 'Original', 'Premium', 'Free delivery',
]
CITIES = ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург', 'Самара', 'Омск', 'Уфа', 'Пермь',
          'Краснодар', 'Воронеж', 'Ростов-на-Дону', 'Нижний Новгород', 'Челябинск', 'Тюмень', 'Барнаул', 'Сочи']
STREETS = ['Ленина', 'Мира', 'Советская', 'Гагарина', 'Пушкина', 'Кирова', 'Садовая', 'Лесная', 'Молодежная',
           'Центральная', 'Школьная', 'Октябрьская', 'Набережная', 'Заводская', 'Строителей', 'Победы']
PLATE_LETTERS = 'АВЕКМНОРСТУХ'

Шрифты: список путей к файлам для трейна или валидации и кэш загруженных шрифтов.
`get_cmap` нужен чтобы знать, какие символы вообще есть в шрифте, и не рисовать квадратики вместо букв.

In [15]:
def load_fonts(split):
    """Список шрифтов для train или val (разбиение по семействам задано в FONTS_TXT выше)."""
    fonts = {}
    for line in FONTS_TXT.strip().splitlines():
        family, category, variants, sp = line.strip().split(';')
        if sp != split:
            continue
        for v in variants.split(','):
            path = os.path.join(FONTS_DIR, f"{family.replace(' ', '_')}-{v}.ttf")
            if os.path.exists(path):
                fonts.setdefault(category, []).append(path)
    return fonts


# загрузка шрифта не быстрая, поэтому кэшируем по (путь, размер)
_font_cache = {}
_cmap_cache = {}


def get_font(path, size):
    key = (path, size)
    if key not in _font_cache:
        _font_cache[key] = ImageFont.truetype(path, size)
    return _font_cache[key]


def get_cmap(path):
    # какие символы есть в шрифте, чтоб не рисовать квадратики вместо букв
    if path not in _cmap_cache:
        _cmap_cache[path] = set(TTFont(path).getBestCmap().keys())
    return _cmap_cache[path]

Тексты. Строки корпуса делим на train и val по хешу (в валидации будут и незнакомые шрифты, и незнакомые тексты).
`TextSampler` выдает случайную строку: чаще короткую (в тесте много одиночных слов), иногда капсом или с большой буквы.

In [16]:
# корпус большой (около миллиона строк), читаем один раз на процесс
_lines_cache = {}


def load_lines(name, split):
    # делим строки на train/val по хешу, чтоб в валидации были и незнакомые шрифты и незнакомые тексты
    if (name, split) in _lines_cache:
        return _lines_cache[(name, split)]
    lines = []
    for l in open(os.path.join(TEXTS_DIR, name), encoding='utf-8'):
        l = l.strip()
        if not l:
            continue
        is_val = zlib.crc32(l.encode('utf-8')) % 10 == 0
        if is_val == (split == 'val'):
            lines.append(l)
    _lines_cache[(name, split)] = lines
    return lines


class TextSampler:
    """Выдает случайные строки текста для синтетики."""

    def __init__(self, split, rng):
        self.rng = rng
        self.ru = load_lines('ru.txt', split)
        self.latin = load_lines('latin.txt', split)

    def digits(self, n):
        return ''.join(str(d) for d in self.rng.integers(0, 10, n))

    def template(self):
        """Всякие телефоны, цены, номера, адреса и тд. Такого на фотках объявлений очень много."""
        r = self.rng
        d = self.digits
        # 12 вариантов: 0-8 разные шаблоны, 9-11 фразы из объявлений (так фраз чуть больше)
        kind = r.integers(0, 12)
        if kind == 0:
            return r.choice([f'+7 (9{d(2)}) {d(3)}-{d(2)}-{d(2)}', f'8 9{d(2)} {d(3)} {d(2)} {d(2)}',
                             f'8-9{d(2)}-{d(3)}-{d(2)}-{d(2)}', f'+79{d(9)}', f'тел. 8 (9{d(2)}) {d(3)}-{d(2)}-{d(2)}',
                             f'8 (800) {d(3)}-{d(2)}-{d(2)}'])
        if kind == 1:
            price = int(r.choice([r.integers(1, 100) * 10, r.integers(1, 300) * 100, r.integers(1, 999) * 1000]))
            p = f'{price:,}'.replace(',', ' ')
            return r.choice([f'{p} ₽', f'{p} руб.', f'{p}р', f'от {p} руб', f'Цена: {p} руб.',
                             f'{p} rub', f'${price}', f'{p},00', f'-{r.integers(5, 70)}%', f'скидка {r.integers(5, 70)}%'])
        if kind == 2:
            l = lambda: r.choice(list(PLATE_LETTERS))
            return f'{l()}{d(3)}{l()}{l()} {r.choice([d(2), d(3)])}'
        if kind == 3:
            return r.choice([f'{r.integers(1, 29):02d}.{r.integers(1, 13):02d}.{r.integers(1995, 2026)}',
                             f'{r.integers(1990, 2026)} г.', f'с {r.integers(7, 11)}:00 до {r.integers(17, 23)}:00',
                             f'ПН-ПТ {r.integers(8, 11)}:00-{r.integers(17, 21)}:00', f'{r.integers(0, 24):02d}:{r.integers(0, 60):02d}'])
        if kind == 4:
            name = ''.join(r.choice(list('abcdefghijklmnopqrstuvwxyz'), r.integers(3, 10)))
            return r.choice([f'www.{name}.ru', f'{name}.com', f'@{name}', f'@{name}_shop', f'{name}@mail.ru',
                             f'https://{name}.ru', f'vk.com/{name}', f't.me/{name}'])
        if kind == 5:
            alnum = list('ABCDEFGHJKLMNPRSTUVWXYZ0123456789')
            code = ''.join(r.choice(alnum, r.integers(5, 17)))
            return r.choice([code, f'SN: {code}', f'S/N {code}', f'Арт. {d(r.integers(4, 9))}', f'арт {d(6)}-{d(2)}',
                             f'ИНН {d(10)}', f'ГОСТ {d(4)}-{r.integers(1990, 2025)}', f'VIN {code}', f'Model: {code[:6]}'])
        if kind == 6:
            return r.choice([f'{r.integers(38, 56)}-{r.integers(40, 58)}', r.choice(['XS', 'S', 'M', 'L', 'XL', 'XXL']),
                             f'{r.integers(10, 300)}x{r.integers(10, 300)} см', f'{r.integers(1, 99)} мм',
                             f'{r.integers(1, 5)}.{r.integers(0, 9)} кВт', f'220V {r.integers(1, 3000)}W',
                             f'{r.integers(1, 99)} кг', f'{r.integers(1, 20)} л', f'{r.integers(16, 1024)} GB'])
        if kind == 7:
            return r.choice([f'г. {r.choice(CITIES)}', f'ул. {r.choice(STREETS)}, д. {r.integers(1, 150)}',
                             f'{r.choice(CITIES)}, ул. {r.choice(STREETS)} {r.integers(1, 99)}',
                             f'пр-т {r.choice(STREETS)} {r.integers(1, 99)}к{r.integers(1, 5)}'])
        if kind == 8:
            return d(r.integers(1, 8))
        return str(r.choice(AD_PHRASES))

    def take_words(self, line):
        # сколько слов берем: чаще короткие куски, в тесте много одиночных слов
        n = int(self.rng.choice([1, 2, 3, 4, 5, 6, 8, 10], p=[.26, .22, .15, .1, .08, .08, .06, .05]))
        words = line.split()
        if len(words) > n:
            start = self.rng.integers(0, len(words) - n + 1)
            words = words[start:start + n]
        text = ' '.join(words)
        return text[:60]

    def sample(self):
        r = self.rng.random()
        # 55% русские строки из корпуса, 17% латинские, остальное шаблоны (телефоны, цены и тд)
        if r < 0.55:
            text = self.take_words(self.ru[self.rng.integers(len(self.ru))])
        elif r < 0.72:
            text = self.take_words(self.latin[self.rng.integers(len(self.latin))])
        else:
            text = self.template()

        # регистр: капсом на вывесках пишут очень часто
        r = self.rng.random()
        if r < 0.28:
            text = text.upper()
        elif r < 0.36:
            text = text.lower()
        elif r < 0.44:
            text = text.title()
        return text

Рисование: цвета с контролем контраста, фон (однотонный, градиент, текстура или из двух частей) и маска текста.
Текст рисуем белым по черному в отдельную маску, а цвет накладываем потом. Так проще делать обводку, тень и градиент.

In [17]:
def luminance(c):
    # яркость цвета как ее видит глаз (стандартные веса для rgb)
    return 0.299 * c[0] + 0.587 * c[1] + 0.114 * c[2]


def random_colors(rng):
    """Цвет текста и фона. Следим чтобы контраст был, но иногда делаем и слабый (такое тоже бывает)."""
    if rng.random() < 0.45:
        # самый частый вариант: темный текст на светлом или наоборот
        dark = rng.integers(0, 70, 3)
        light = rng.integers(170, 256, 3)
        return (dark, light) if rng.random() < 0.65 else (light, dark)
    min_contrast = 25 if rng.random() < 0.15 else 60
    for _ in range(50):
        fg = rng.integers(0, 256, 3)
        bg = rng.integers(0, 256, 3)
        if abs(luminance(fg) - luminance(bg)) >= min_contrast:
            return fg, bg
    return np.array([0, 0, 0]), np.array([255, 255, 255])


def make_background(h, w, color, rng):
    """Фон: однотонный, градиент, шумная текстура или из двух частей."""
    color = color.astype(np.float32)
    kind = rng.choice(['solid', 'gradient', 'noise', 'split'], p=[0.45, 0.25, 0.2, 0.1])
    bg = np.empty((h, w, 3), np.float32)
    bg[:] = color
    if kind == 'gradient':
        other = np.clip(color + rng.normal(0, 40, 3), 0, 255)
        if rng.random() < 0.5:
            t = np.linspace(0, 1, w)[None, :, None]
        else:
            t = np.linspace(0, 1, h)[:, None, None]
        bg = (bg * (1 - t) + other * t).astype(np.float32)
    elif kind == 'noise':
        # низкочастотный шум: маленькую случайную картинку растягиваем до нужного размера
        small = rng.normal(0, rng.uniform(10, 40), (max(2, h // 8), max(2, w // 8), 3)).astype(np.float32)
        bg = bg + cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)
    elif kind == 'split':
        other = np.clip(color + rng.normal(0, 50, 3), 0, 255)
        if rng.random() < 0.5:
            bg[:, int(rng.integers(0, w)):] = other
        else:
            bg[int(rng.integers(0, h)):, :] = other
    if rng.random() < 0.3:
        # чуть мелкого шума чтоб фон не был идеально гладким
        bg += rng.standard_normal(bg.shape, dtype=np.float32) * rng.uniform(1, 6)
    return bg


def draw_line_mask(text, font, spacing=0.0, stroke=0):
    """Рисует строку белым по черному, возвращает маску (uint8) обрезанную по тексту."""
    if spacing == 0:
        l, t, r, b = font.getbbox(text, stroke_width=stroke)
        img = Image.new('L', (r - l + 4, b - t + 4), 0)
        ImageDraw.Draw(img).text((2 - l, 2 - t), text, font=font, fill=255, stroke_width=stroke, stroke_fill=255)
    else:
        # рисуем по одной букве с доп. расстоянием (разрядка, бывает на вывесках)
        size = font.size
        widths = [font.getlength(ch) for ch in text]
        total = int(sum(widths) + spacing * size * len(text) + size + 4)
        img = Image.new('L', (total, int(size * 2)), 0)
        d = ImageDraw.Draw(img)
        x = 2
        for ch, cw in zip(text, widths):
            d.text((x, size // 2), ch, font=font, fill=255, stroke_width=stroke, stroke_fill=255)
            x += cw + spacing * size
    arr = np.array(img)
    ys, xs = np.nonzero(arr > 0)
    if len(ys) == 0:
        return None
    return arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

Сам генератор. `_sample` собирает одну картинку по шагам из описания выше, `crop_box` режет бокс со случайными полями,
`degrade` портит качество. Если что то пошло не так (например в шрифте нет нужных букв), пробуем еще раз.

In [18]:
class SynthGenerator:
    """Генератор синтетических боксов с текстом. На выходе картинка после препроцессинга (32, w) и сам текст."""

    def __init__(self, split, seed):
        self.rng = np.random.default_rng(seed)
        self.fonts = load_fonts(split)
        self.categories = [c for c in CATEGORY_P if c in self.fonts]
        p = np.array([CATEGORY_P[c] for c in self.categories])
        # нормируем вероятности категорий (вдруг какой то категории нет среди скачанных шрифтов)
        self.cat_p = p / p.sum()
        self.texts = TextSampler(split, self.rng)

    def pick_font(self):
        cat = self.rng.choice(self.categories, p=self.cat_p)
        paths = self.fonts[cat]
        return paths[self.rng.integers(len(paths))]

    def pick_text(self, font_path):
        # выкидываем символы которых нет в шрифте. Если выкинули много, то берем другой текст
        cmap = get_cmap(font_path)
        for _ in range(10):
            text = self.texts.sample()
            clean = ''.join(ch for ch in text if ch == ' ' or ord(ch) in cmap).strip()
            if clean and len(clean) >= 0.8 * len(text):
                return clean
        return None

    def render_block(self, font, texts, main_idx, spacing, stroke, line_gap):
        """
        Рисует основную строку и (если есть) соседние строки над и под ней.
        Возвращает маску заливки, маску с обводкой и маску основной строки (по ней потом режем бокс).
        """
        masks = []
        for t in texts:
            m_fill = draw_line_mask(t, font, spacing, 0)
            if m_fill is None:
                return None
            m_stroke = draw_line_mask(t, font, spacing, stroke) if stroke else m_fill
            masks.append((m_fill, m_stroke))
        pad = stroke + 2
        width = max(ms.shape[1] for _, ms in masks) + 2 * pad
        heights = [ms.shape[0] for _, ms in masks]
        gap = int(line_gap * font.size)
        height = sum(heights) + gap * (len(masks) - 1) + 2 * pad
        fill = np.zeros((height, width), np.uint8)
        outline = np.zeros((height, width), np.uint8)
        main = np.zeros((height, width), np.uint8)
        y = pad
        for i, (mf, ms) in enumerate(masks):
            # выравнивание строк: по левому краю или по центру
            x = pad if self.rng.random() < 0.5 else (width - ms.shape[1]) // 2
            dy = (ms.shape[0] - mf.shape[0]) // 2
            dx = (ms.shape[1] - mf.shape[1]) // 2
            outline[y:y + ms.shape[0], x:x + ms.shape[1]] = np.maximum(outline[y:y + ms.shape[0], x:x + ms.shape[1]], ms)
            fill[y + dy:y + dy + mf.shape[0], x + dx:x + dx + mf.shape[1]] = mf
            if i == main_idx:
                main[y:y + ms.shape[0], x:x + ms.shape[1]] = ms
            y += ms.shape[0] + gap
        return fill, outline, main

    def sample(self):
        for _ in range(20):
            out = self._sample()
            if out is not None:
                return out
        raise RuntimeError('не получилось сгенерить картинку')

    def _sample(self):
        rng = self.rng
        font_path = self.pick_font()
        text = self.pick_text(font_path)
        if text is None:
            return None
        # размер шрифта в пикселях. Потом все равно ужимаем до высоты 32, а разные размеры дают разное сглаживание букв
        size = int(rng.choice([18, 22, 26, 30, 36, 42, 48]))
        font = get_font(font_path, size)

        # разрядка букв (12% картинок) и обводка (15%)
        spacing = rng.uniform(0.05, 0.4) if rng.random() < 0.12 else 0.0
        stroke = int(rng.integers(1, max(2, size // 10) + 1)) if rng.random() < 0.15 else 0

        # иногда рисуем соседние строки, потом при обрезке от них останутся куски сверху/снизу
        texts, main_idx = [text], 0
        r = rng.random()
        if r < 0.12:
            texts, main_idx = [self.pick_text(font_path), text], 1
        elif r < 0.2:
            texts, main_idx = [text, self.pick_text(font_path)], 0
        elif r < 0.26:
            texts, main_idx = [self.pick_text(font_path), text, self.pick_text(font_path)], 1
        if any(t is None for t in texts):
            texts, main_idx = [text], 0
        block = self.render_block(font, texts, main_idx, spacing, stroke, line_gap=rng.uniform(0.15, 0.6))
        if block is None:
            return None
        fill, outline, main = block

        # растяжение/сжатие по горизонтали (узкие и широкие начертания)
        if rng.random() < 0.3:
            k = rng.uniform(0.7, 1.35)
            new_w = max(4, int(fill.shape[1] * k))
            fill, outline, main = [cv2.resize(m, (new_w, m.shape[0]), interpolation=cv2.INTER_LINEAR)
                                   for m in (fill, outline, main)]
        # наклон как у курсива
        if rng.random() < 0.12:
            fill, outline, main = self.shear([fill, outline, main], rng.uniform(-0.3, 0.3))

        # добавляем поля, чтобы было куда вращать и откуда резать бокс
        m = size // 2 + 2
        fill, outline, main = [cv2.copyMakeBorder(x, m, m, m, m, cv2.BORDER_CONSTANT, value=0)
                               for x in (fill, outline, main)]
        h, w = fill.shape

        # ---- собираем цветную картинку ----
        # цвет текста и фона, потом сам фон
        fg, bgc = random_colors(rng)
        img = make_background(h, w, bgc, rng)

        if rng.random() < 0.08:
            # плашка под текстом (цветной прямоугольник)
            plate = np.clip(bgc + rng.normal(0, 60, 3), 0, 255)
            ys, xs = np.nonzero(main)
            y0, y1 = ys.min() - int(rng.integers(0, m // 2 + 1)), ys.max() + int(rng.integers(0, m // 2 + 1))
            x0, x1 = xs.min() - int(rng.integers(0, m + 1)), xs.max() + int(rng.integers(0, m + 1))
            img[max(0, y0):y1, max(0, x0):x1] = plate

        # маски переводим в [0, 1], по ним смешиваем цвет текста с фоном
        a_fill = fill.astype(np.float32)[..., None] / 255
        a_out = outline.astype(np.float32)[..., None] / 255
        fg = fg.astype(np.float32)
        if rng.random() < 0.12:
            # тень: сдвинутая размытая копия текста, чаще вниз-вправо (свет обычно сверху)
            dx, dy = int(rng.integers(-1, 4)), int(rng.integers(0, 4))
            sh = np.roll(np.roll(outline, dy, axis=0), dx, axis=1).astype(np.float32)
            sh = cv2.GaussianBlur(sh, (0, 0), rng.uniform(0.5, 2.0))[..., None] / 255
            img = img * (1 - sh * 0.8) + np.float32(20) * sh * 0.8
        if stroke:
            stroke_color = rng.integers(0, 256, 3).astype(np.float32)
            img = img * (1 - a_out) + stroke_color * a_out
        # иногда текст не одним цветом а с легким градиентом
        if rng.random() < 0.15:
            text_color = fg + np.linspace(-40, 40, h, dtype=np.float32)[:, None, None] * rng.choice([-1, 1])
        else:
            text_color = fg
        img = img * (1 - a_fill) + text_color * a_fill

        if rng.random() < 0.05:
            # подчеркивание под основной строкой
            ys, xs = np.nonzero(main)
            yy = min(h - 1, ys.max() + int(rng.integers(1, 4)))
            img[yy:yy + max(1, size // 15), xs.min():xs.max()] = fg

        img = np.clip(img, 0, 255).astype(np.uint8)

        # ---- геометрия: небольшой поворот / перспектива ----
        img, main = self.warp(img, main)

        # ---- вырезаем бокс вокруг основной строки ----
        crop = self.crop_box(img, main)
        if crop is None:
            return None

        # ---- портим качество ----
        crop = self.degrade(crop)
        return prepare_image(crop), text

    def shear(self, masks, k):
        h, w = masks[0].shape
        shift = abs(k) * h
        mat = np.float32([[1, k, shift if k < 0 else 0], [0, 1, 0]])
        new_w = int(w + shift) + 1
        return [cv2.warpAffine(x, mat, (new_w, h)) for x in masks]

    def warp(self, img, main):
        rng = self.rng
        h, w = main.shape
        if rng.random() < 0.5:
            angle = rng.normal(0, 1.5)
            mat = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
            img = cv2.warpAffine(img, mat, (w, h), borderMode=cv2.BORDER_REPLICATE)
            main = cv2.warpAffine(main, mat, (w, h))
        if rng.random() < 0.25:
            d = 0.06 * min(h, w)
            src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
            dst = src + rng.uniform(-d, d, (4, 2)).astype(np.float32)
            mat = cv2.getPerspectiveTransform(src, dst)
            img = cv2.warpPerspective(img, mat, (w, h), borderMode=cv2.BORDER_REPLICATE)
            main = cv2.warpPerspective(main, mat, (w, h))
        return img, main

    def crop_box(self, img, main):
        """Режем прямоугольник вокруг основной строки со случайными полями (как у неидеального детектора)."""
        rng = self.rng
        ys, xs = np.nonzero(main > 64)
        if len(ys) == 0:
            return None
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        th = y1 - y0
        # поля сверху и снизу независимые и с одинаковым распределением, значит никакой подсказки про ориентацию
        top = int(th * rng.uniform(-0.08, 0.35))
        bottom = int(th * rng.uniform(-0.08, 0.35))
        left = int(th * rng.uniform(-0.05, 0.6))
        right = int(th * rng.uniform(-0.05, 0.6))
        H, W = img.shape[:2]
        y0, y1 = max(0, y0 - top), min(H, y1 + bottom)
        x0, x1 = max(0, x0 - left), min(W, x1 + right)
        if y1 - y0 < 4 or x1 - x0 < 4:
            return None
        return img[y0:y1, x0:x1]

    def degrade(self, img):
        """Уменьшаем до случайной высоты (в тесте много мелких картинок), блюр, шум, jpeg."""
        rng = self.rng
        # высота как в тесте: от ~12 до ~100 пикселей, чаще маленькие
        target_h = int(np.exp(rng.uniform(np.log(12), np.log(100))))
        h, w = img.shape[:2]
        if target_h < h:
            new_w = max(4, int(w * target_h / h))
            img = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)
        if rng.random() < 0.35:
            img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, 1.2))
        if rng.random() < 0.08:
            # смаз от движения по горизонтали
            k = int(rng.integers(2, 5))
            kernel = np.zeros((k, k), np.float32)
            kernel[k // 2, :] = 1.0 / k
            img = cv2.filter2D(img, -1, kernel)
        img = img.astype(np.float32)
        if rng.random() < 0.4:
            # яркость / контраст
            img = img * rng.uniform(0.6, 1.3) + rng.uniform(-40, 40)
        if rng.random() < 0.15:
            # неравномерное освещение
            g = np.linspace(rng.uniform(0.6, 1.0), rng.uniform(0.9, 1.3), img.shape[1])[None, :, None]
            img = img * g
        if rng.random() < 0.4:
            img = img + rng.normal(0, rng.uniform(2, 12), img.shape)
        img = np.clip(img, 0, 255).astype(np.uint8)
        if rng.random() < 0.5:
            q = int(rng.integers(15, 90))
            ok, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, q])
            if ok:
                img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        return img

Генерация пачки. Картинки генерим кусками по 2000 штук, у каждого куска свой сид, поэтому результат не зависит от числа процессов
и повторяется от запуска к запуску.

In [19]:
CHUNK = 2000


def _gen_chunk(args):
    """Генерит n картинок одним генератором с заданным сидом (выполняется в отдельном процессе)."""
    split, seed, n = args
    gen = SynthGenerator(split, seed)
    arrs, texts = [], []
    for _ in range(n):
        a, t = gen.sample()
        arrs.append(a)
        texts.append(t)
    return arrs, texts


def generate_synth(split, n, out, seed=0, workers=4):
    """Генерит n синтетических картинок для split ('train' или 'val') и сохраняет пачкой в out."""
    # у каждого куска свой сид, поэтому результат не зависит от числа процессов
    n_chunks = (n + CHUNK - 1) // CHUNK
    jobs = [(split, seed * 100000 + i + (50000 if split == 'val' else 0),
             min(CHUNK, n - i * CHUNK)) for i in range(n_chunks)]
    arrs, texts = [], []
    if workers > 1 and sys.platform.startswith('linux'):
        # функции объявлены прямо в ноутбуке, поэтому процессы только через fork (он есть на линуксе)
        with get_context('fork').Pool(workers) as pool:
            for a, t in tqdm(pool.imap(_gen_chunk, jobs), total=len(jobs)):
                arrs += a
                texts += t
    else:
        for job in tqdm(jobs):
            a, t = _gen_chunk(job)
            arrs += a
            texts += t
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    save_pack(out, arrs, texts=np.array(texts))
    print('сохранил', len(arrs), 'картинок в', os.path.relpath(out, ROOT))

In [20]:
if TRAIN:
    # train: 300 тысяч картинок (минут 10-15 на 4 ядрах), val: 10 тысяч с другими шрифтами и текстами
    for split, n in [('train', 300000), ('val', 10000)]:
        out = os.path.join(SYNTH_DIR, f'{split}.npz')
        if os.path.exists(out):
            print(os.path.relpath(out, ROOT), 'уже есть, пропускаю')
        else:
            generate_synth(split, n, out)
else:
    print('TRAIN = False, синтетика не нужна')

TRAIN = False, синтетика не нужна


## 5. Датасет: поворот на 180 на лету

Все картинки в пачках стоят правильно и уже приведены к высоте 32. Метку делаем сами:
с вероятностью 0.5 переворачиваем картинку на 180 и ставим label = 1, иначе label = 0.
Поворот делаем до паддинга, так же как (скорее всего) повернут тест.

На обучении к картинкам еще добавляются легкие аугментации (прямо на картинке 32 x w, это дешево).
Все они симметричные относительно поворота, то есть не подсказывают сетке ответ.
На валидации поворот фиксированный (по сиду) и без аугментаций, чтобы метрики между запусками можно было сравнивать.

In [21]:
def load_packs(names, folder):
    """Открывает пачки картинок data/<folder>/<name>.npz."""
    return [Pack(os.path.join(ROOT, 'data', folder, n + '.npz')) for n in names]


def light_aug(arr, rng):
    """
    Легкие аугментации прямо на картинке 32 x w (дешево, можно на лету).
    Все они симметричные относительно поворота, т.е. не подсказывают сетке ответ.
    """
    h, w = arr.shape
    # обрезаем немного слева/справа (детектор мог отрезать кусок текста)
    if rng.random() < 0.3 and w > 16:
        l = int(rng.integers(0, int(w * 0.15) + 1))
        r = int(rng.integers(0, int(w * 0.15) + 1))
        arr = arr[:, l:w - r] if w - r - l > 8 else arr
    # немного сжимаем/растягиваем по ширине
    if rng.random() < 0.3:
        k = rng.uniform(0.8, 1.25)
        arr = cv2.resize(arr, (max(4, int(arr.shape[1] * k)), h), interpolation=cv2.INTER_LINEAR)
    # сдвиг по вертикали на пару пикселей (срезаем сверху или снизу и растягиваем обратно)
    if rng.random() < 0.2:
        cut = int(rng.integers(1, 4))
        arr = arr[cut:] if rng.random() < 0.5 else arr[:-cut]
        arr = cv2.resize(arr, (arr.shape[1], h), interpolation=cv2.INTER_LINEAR)
    # инверсия (светлый текст на темном становится темным на светлом и наоборот)
    if rng.random() < 0.2:
        arr = 255 - arr
    # гамма
    if rng.random() < 0.3:
        # гамма меняет контраст нелинейно, как разные камеры и экраны
        g = rng.uniform(0.6, 1.6)
        arr = (255 * (arr / 255.0) ** g).astype(np.uint8)
    # блюр и шум
    if rng.random() < 0.1:
        arr = cv2.GaussianBlur(arr, (0, 0), rng.uniform(0.4, 1.0))
    if rng.random() < 0.2:
        noise = rng.normal(0, rng.uniform(3, 12), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return np.ascontiguousarray(arr)

In [22]:
class RotDataset(Dataset):
    """
    Склеивает несколько пачек в один датасет.
    train=True: случайный поворот + легкие аугментации
    train=False: поворот фиксирован сидом, без аугментаций
    """

    def __init__(self, packs, width, train=True, seed=0):
        self.packs = packs
        self.width = width
        self.train = train
        # глобальный индекс превращаем в пару (номер пачки, индекс внутри пачки)
        self.index = [(pi, i) for pi, p in enumerate(packs) for i in range(len(p))]
        self.seed = seed
        rng = np.random.RandomState(seed)
        # для валидации заранее решаем какие картинки повернуты, чтобы это не менялось между эпохами и запусками
        self.fixed_rot = rng.rand(len(self.index)) < 0.5
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        pi, i = self.index[idx]
        arr = self.packs[pi][i]
        if self.train:
            arr = light_aug(arr, self.rng)
            # на трейне монетка: повернуть или нет, это и есть метка
            rot = self.rng.random() < 0.5
        else:
            rot = self.fixed_rot[idx]
        if rot:
            arr = rotate180(arr)
        # дальше тот же препроцессинг что и на тесте
        x = to_model_input(arr, self.width)
        return torch.from_numpy(x), torch.tensor(float(rot))


def worker_init(worker_id):
    # у каждого воркера даталоадера свой генератор, иначе аугментации будут одинаковые
    info = torch.utils.data.get_worker_info()
    ds = info.dataset
    ds.rng = np.random.default_rng(ds.seed * 1000 + worker_id + 1 + torch.initial_seed() % 100000)


def mix_weights(packs_a, packs_b, share_b):
    """
    Веса для семплера: чтоб доля картинок из packs_b (реальные кропы) в батчах была примерно share_b.
    Реальных кропов в 4.5 раза меньше чем синтетики, без весов их было бы всего 18%.
    """
    na = sum(len(p) for p in packs_a)
    nb = sum(len(p) for p in packs_b)
    wa = (1 - share_b) / na
    wb = share_b / nb
    return torch.tensor([wa] * na + [wb] * nb, dtype=torch.double)

## 6. Модель TinyCNN

Устроена как сверточная часть CRNN из OCR, только без рекуррентной части и сильно меньше:
* несколько блоков свертка 3x3 + batchnorm + relu, между ними max pooling. По высоте картинка постепенно сжимается с 32 до 2
* последняя свертка сразу на всю оставшуюся высоту (2 пикселя), тут сетка видит одновременно верх и низ строки.
  Для ориентации это как раз главное: хвосты букв вверх и вниз, точки и запятые внизу, заглавные буквы слева и тд
* по ширине просто усредняем (global average pooling), так сетке все равно какой ширины картинка
* линейный слой выдает один логит: больше нуля значит скорее перевернуто

Каналы 16, 32, 64, 96, 128, всего 273 тысячи параметров. Ширину можно менять множителем `width_mult`,
сеть в 2 раза уже (w0.5) я тоже пробовал, она заметно хуже (см. описание подхода).

In [23]:
def conv_bn(cin, cout, k=3, stride=1, padding=None):
    """Свертка + batchnorm + relu, базовый блок сети."""
    if padding is None:
        padding = k // 2 if isinstance(k, int) else (k[0] // 2, k[1] // 2)
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, stride, padding, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class TinyCNN(nn.Module):
    def __init__(self, width_mult=1.0):
        super().__init__()
        # число каналов на каждом этапе (при width_mult = 1.0 это 16, 32, 64, 96, 128)
        c = [max(8, int(round(x * width_mult))) for x in (16, 32, 64, 96, 128)]
        self.features = nn.Sequential(
            conv_bn(1, c[0]),                  # 32 x W
            nn.MaxPool2d(2),                   # 16 x W/2
            conv_bn(c[0], c[1]),
            nn.MaxPool2d(2),                   # 8 x W/4
            conv_bn(c[1], c[2]),
            conv_bn(c[2], c[2]),
            nn.MaxPool2d(2),                   # 4 x W/8
            conv_bn(c[2], c[3]),
            conv_bn(c[3], c[3]),
            nn.MaxPool2d((2, 1)),              # 2 x W/8
            # последняя свертка на всю оставшуюся высоту, после нее 1 x W/8.
            # тут сетка видит сразу и верх и низ строки, это как раз то что нужно для ориентации
            conv_bn(c[3], c[4], k=(2, 3), padding=(0, 1)),
        )
        # немного dropout перед последним слоем, чтобы меньше переобучаться на синтетику
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(c[4], 1)

    def forward(self, x):
        f = self.features(x)          # B x C x 1 x W/8
        f = f.mean(dim=(2, 3))        # B x C
        return self.head(self.dropout(f)).squeeze(1)


def build_model(cfg):
    return TinyCNN(width_mult=cfg['model'].get('width_mult', 1.0))


def count_params(model):
    return sum(p.numel() for p in model.parameters())

## 7. Метрики и обучение

Основная метрика задачи: 1 - Brier (`score`). Кроме нее смотрю logloss, accuracy и ECE (expected calibration error:
насколько предсказанные вероятности совпадают с реальной частотой, для Brier это важно).

In [24]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def metrics(y, p):
    """Все метрики сразу. y: настоящие метки 0/1, p: предсказанные вероятности."""
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    brier = np.mean((p - y) ** 2)
    pc = np.clip(p, 1e-6, 1 - 1e-6)
    logloss = -np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc))
    acc = np.mean((p > 0.5) == (y > 0.5))
    return {'score': 1 - brier, 'brier': brier, 'logloss': logloss, 'acc': acc, 'ece': ece(y, p)}


def ece(y, p, bins=15):
    # expected calibration error: разбиваем на корзины по уверенности и смотрим разницу с реальной частотой
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    err = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            err += m.mean() * abs(p[m].mean() - y[m].mean())
    return err


@torch.no_grad()
def model_logits(model, loader):
    """Логиты торчевой модели и метки для всего даталоадера."""
    model.eval()
    logits, labels = [], []
    for x, y in loader:
        logits.append(model(x).numpy())
        labels.append(y.numpy())
    return np.concatenate(logits), np.concatenate(labels)

Обучение:
* трейн: синтетика + реальные кропы, 30% реальных в батче. Картинки семплируем с возвращением, 150 тысяч на эпоху
* лосс: BCE с label smoothing 0.02 (цель 0.01 и 0.99 вместо 0 и 1). Часть картинок в принципе неразличима
  ("ОНО", "88", размытое пятно), так сетка не учится быть на них слишком уверенной
* оптимизатор AdamW, расписание OneCycle (сначала разгон lr, потом плавное затухание)
* после каждой эпохи валидация по всем источникам. Лучшую эпоху выбираю по среднему скору на синтетической и на реальной валидации
* channels_last: на cpu это ускоряет свертки примерно на треть
* все сохраняется в `runs/<name>/`: веса лучшей и последней эпохи, история метрик, настройки

In [25]:
def make_val_loaders(cfg, width):
    """Отдельный даталоадер на каждый источник валидации, чтобы видеть метрики по каждому."""
    loaders = {}
    for name in cfg['data']['synth_val']:
        loaders['synth_' + name] = load_packs([name], 'synth')
    for name in cfg['data']['real_val']:
        loaders[name] = load_packs([name], 'real')
    return {k: DataLoader(RotDataset(v, width, train=False, seed=1), batch_size=512, num_workers=0)
            for k, v in loaders.items()}


def validate(model, val_loaders):
    """Метрики по каждому источнику + по всем реальным вместе + критерий выбора лучшей эпохи (select)."""
    res = {}
    all_real_y, all_real_p = [], []
    for name, loader in val_loaders.items():
        logits, y = model_logits(model, loader)
        p = sigmoid(logits)
        res[name] = metrics(y, p)
        if not name.startswith('synth'):
            all_real_y.append(y)
            all_real_p.append(p)
    res['real_all'] = metrics(np.concatenate(all_real_y), np.concatenate(all_real_p))
    synth_keys = [k for k in res if k.startswith('synth')]
    # select: среднее скоров на синтетике и на всех реальных, по нему выбираем лучшую эпоху
    res['select'] = (np.mean([res[k]['score'] for k in synth_keys]) + res['real_all']['score']) / 2
    return res

In [26]:
def train(cfg, name):
    """Обучение с нуля. Результат в runs/<name>/: best.pt, last.pt, history.json, config.json."""
    seed_everything(cfg['seed'])
    tc = cfg['train']
    torch.set_num_threads(tc['threads'])
    out_dir = os.path.join(ROOT, 'runs', name)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, 'config.json'), 'w') as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)

    # ---- данные ----
    width = cfg['input']['width']
    synth = load_packs(cfg['data']['synth_train'], 'synth')
    real = load_packs(cfg['data']['real_train'], 'real')
    train_ds = RotDataset(synth + real, width, train=True, seed=cfg['seed'])
    # веса для семплера, чтобы в батче было 30% реальных кропов
    weights = mix_weights(synth, real, cfg['data']['real_share'])
    gen = torch.Generator().manual_seed(cfg['seed'])
    sampler = WeightedRandomSampler(weights, num_samples=tc['samples_per_epoch'], replacement=True, generator=gen)
    train_loader = DataLoader(train_ds, batch_size=tc['batch_size'], sampler=sampler, num_workers=tc['workers'],
                              worker_init_fn=worker_init, drop_last=True, persistent_workers=tc['workers'] > 0)
    val_loaders = make_val_loaders(cfg, width)
    print(f'трейн: синтетика {sum(map(len, synth))}, реальные {sum(map(len, real))}')

    # ---- модель и оптимизатор ----
    model = build_model(cfg)
    # channels_last на cpu дает заметное ускорение сверток (проверял, примерно на треть)
    model = model.to(memory_format=torch.channels_last)
    print('параметров:', count_params(model))

    steps = tc['epochs'] * len(train_loader)
    opt = torch.optim.AdamW(model.parameters(), lr=tc['lr'], weight_decay=tc['weight_decay'])
    # OneCycle: первые 15% шагов lr растет, дальше плавно падает почти до нуля
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=tc['lr'], total_steps=steps, pct_start=0.15)
    ls = tc['label_smoothing']

    # ---- цикл по эпохам ----
    best, history = -1, []
    for epoch in range(tc['epochs']):
        model.train()
        t0 = time.time()
        losses = []
        for step, (x, y) in enumerate(train_loader):
            x = x.to(memory_format=torch.channels_last)
            target = y * (1 - ls) + 0.5 * ls  # label smoothing
            loss = F.binary_cross_entropy_with_logits(model(x), target)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
            if step % 100 == 0:
                print(f'  эпоха {epoch} шаг {step}/{len(train_loader)} loss {np.mean(losses[-100:]):.4f}', flush=True)

        # после каждой эпохи валидация по всем источникам
        res = validate(model, val_loaders)
        res['epoch'] = epoch
        res['train_loss'] = float(np.mean(losses))
        res['time'] = time.time() - t0
        history.append(res)
        line = ' | '.join(f"{k} {v['score']:.4f}" for k, v in res.items() if isinstance(v, dict))
        print(f'эпоха {epoch}: loss {res["train_loss"]:.4f}, {line}, select {res["select"]:.4f}, {res["time"]:.0f}с', flush=True)

        torch.save(model.state_dict(), os.path.join(out_dir, 'last.pt'))
        # сохраняем веса лучшей эпохи
        if res['select'] > best:
            best = res['select']
            torch.save(model.state_dict(), os.path.join(out_dir, 'best.pt'))
        with open(os.path.join(out_dir, 'history.json'), 'w') as f:
            json.dump(history, f, indent=2)
    print('лучший select:', best)

In [27]:
if TRAIN:
    train(CFG, 'base_w1')  # около 80 минут на cpu с 4 ядрами, лучшая эпоха в runs/base_w1/best.pt
else:
    print('TRAIN = False, беру готовые веса', os.path.relpath(WEIGHTS, ROOT))

TRAIN = False, беру готовые веса weights/model.onnx


## 8. Экспорт в onnx и замер скорости

В проде модель будет работать на каждом боксе каждой картинки, поэтому скорость важна.
Экспортируем в onnx (onnxruntime на cpu заметно быстрее торча), проверяем что onnx выдает то же самое что торч,
и меряем сколько миллисекунд уходит на один бокс на ОДНОМ ядре cpu: по одной картинке (батч 1) и пачкой (батч 64).

In [28]:
def export(weights, out, width=256):
    """Загружает торчевые веса и сохраняет модель в onnx (с любым размером батча)."""
    cfg = json.load(open(os.path.join(os.path.dirname(weights), 'config.json')))
    model = build_model(cfg)
    model.load_state_dict(torch.load(weights, map_location='cpu'))
    model.eval()
    x = torch.randn(1, 1, 32, width)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    torch.onnx.export(model, x, out, input_names=['image'], output_names=['logit'],
                      dynamic_axes={'image': {0: 'batch'}, 'logit': {0: 'batch'}}, opset_version=17, dynamo=False)
    return model


def benchmark(fn, x, n_warmup=5, n_iter=30):
    """Сколько миллисекунд занимает один вызов fn(x)."""
    for _ in range(n_warmup):
        fn(x)
    t = time.perf_counter()
    for _ in range(n_iter):
        fn(x)
    return (time.perf_counter() - t) / n_iter * 1000


def speed(onnx_path, width=256, threads=1):
    """Миллисекунды на один бокс на одном ядре cpu, для батча 1 и батча 64."""
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    sess = ort.InferenceSession(onnx_path, so, providers=['CPUExecutionProvider'])
    name = sess.get_inputs()[0].name
    res = {}
    for bs in (1, 64):
        x = np.random.randn(bs, 1, 32, width).astype(np.float32)
        ms = benchmark(lambda a: sess.run(None, {name: a}), x, n_iter=50 if bs == 1 else 10)
        res[f'ms_per_box_bs{bs}'] = ms / bs
    return res


def export_model(weights, out, width=256):
    """Экспорт, проверка что onnx совпадает с торчем, замер скорости. Результат замера пишем рядом с моделью."""
    model = export(weights, out, width)

    # проверяем что onnx выдает то же самое что торч
    x = np.random.randn(8, 1, 32, width).astype(np.float32)
    sess = ort.InferenceSession(out, providers=['CPUExecutionProvider'])
    y_onnx = sess.run(None, {'image': x})[0]
    with torch.no_grad():
        y_torch = model(torch.from_numpy(x)).numpy()
    print('макс. разница onnx vs torch:', float(np.abs(y_onnx - y_torch).max()))

    info = {'params': count_params(model), 'size_kb': os.path.getsize(out) / 1024}
    info.update(speed(out, width))
    print(json.dumps(info, indent=2))
    with open(os.path.splitext(out)[0] + '_info.json', 'w') as f:
        json.dump(info, f, indent=2)

In [29]:
if TRAIN:
    export_model(os.path.join(ROOT, 'runs', 'base_w1', 'best.pt'), WEIGHTS)
else:
    # веса уже готовые, просто показываем размер и скорость
    info = {'size_kb': os.path.getsize(WEIGHTS) / 1024}
    info.update(speed(WEIGHTS, CFG['input']['width']))
    print(json.dumps(info, indent=2))

{
  "size_kb": 1069.62109375,
  "ms_per_box_bs1": 0.8703333399989788,
  "ms_per_box_bs64": 0.885797160937507
}


## 9. Предикт и submission.csv

Картинки из `data/test/images`, порядок id как в `data/sample_submission.csv`.
Препроцессинг тот же что и в обучении (раздел 3), вероятность это просто sigmoid от логита сети.

In [30]:
def load_predictor(path, threads=4):
    """Возвращает функцию, которая по батчу (N, 1, 32, W) в numpy выдает логиты (N,). Умеет onnx и торчевые .pt"""
    if path.endswith('.onnx'):
        import onnxruntime as ort
        so = ort.SessionOptions()
        so.intra_op_num_threads = threads
        sess = ort.InferenceSession(path, so, providers=['CPUExecutionProvider'])
        name = sess.get_inputs()[0].name
        return lambda x: sess.run(None, {name: x})[0].reshape(-1)

    torch.set_num_threads(threads)
    cfg = json.load(open(os.path.join(os.path.dirname(path), 'config.json')))
    model = build_model(cfg)
    model.load_state_dict(torch.load(path, map_location='cpu'))
    model.eval()

    def run(x):
        with torch.no_grad():
            return model(torch.from_numpy(x)).numpy()
    return run


def load_test(img_dir, ids, width):
    """Читает тестовые картинки и сразу готовит входы для сетки (тот же препроцессинг что в обучении)."""
    xs = []
    for i in tqdm(ids, desc='читаю картинки'):
        arr = prepare_image(read_image(os.path.join(img_dir, i + '.png')))
        xs.append(to_model_input(arr, width))
    return np.stack(xs)


def predict_logits(predictor, x, batch_size=256):
    """Прогоняем все входы через модель батчами."""
    out = []
    for s in tqdm(range(0, len(x), batch_size), desc='предикт'):
        out.append(predictor(x[s:s + batch_size]))
    return np.concatenate(out)


def make_submission(weights, out, width=256):
    """Предикт для всех тестовых картинок и запись submission.csv."""
    ids = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))['image_id'].tolist()
    x = load_test(os.path.join(DATA_DIR, 'test', 'images'), ids, width)
    p = sigmoid(predict_logits(load_predictor(weights), x))

    sub = pd.DataFrame({'image_id': ids, 'p_180': p})
    sub.to_csv(out, index=False)
    print(f'сохранил {os.path.relpath(out, ROOT)}: {len(sub)} строк, среднее p_180 = {p.mean():.4f}')
    return sub

### Проверка готовой модели на валидации

Прежде чем делать сабмит, смотрим метрики финальной модели (`weights/model.onnx`) на той же валидации что и при обучении:
синтетика с незнакомыми шрифтами и текстами + реальные кропы, у каждого источника свой фиксированный поворот (сид 1).
`real_all` это все реальные кропы вместе. Нужны данные из раздела 4, если их нет (запуск с `TRAIN = False` на чистой машине),
ячейка просто скажет об этом.

In [31]:
def evaluate_weights(weights, cfg):
    """Метрики готовой onnx модели на всей валидации, ровно как в train()."""
    predictor = load_predictor(weights)
    width = cfg['input']['width']
    names = [('synth_' + n, n, 'synth') for n in cfg['data']['synth_val']]
    names += [(n, n, 'real') for n in cfg['data']['real_val']]
    rows, real_y, real_p = {}, [], []
    for key, name, folder in names:
        ds = RotDataset(load_packs([name], folder), width, train=False, seed=1)
        x = np.stack([ds[i][0].numpy() for i in range(len(ds))])
        y = ds.fixed_rot.astype(np.float64)
        p = sigmoid(predict_logits(predictor, x))
        rows[key] = metrics(y, p)
        if folder == 'real':
            real_y.append(y)
            real_p.append(p)
    rows['real_all'] = metrics(np.concatenate(real_y), np.concatenate(real_p))
    return pd.DataFrame(rows).T[['score', 'acc', 'logloss', 'ece']]


val_files = [os.path.join(SYNTH_DIR, n + '.npz') for n in CFG['data']['synth_val']]
val_files += [os.path.join(REAL_DIR, n + '.npz') for n in CFG['data']['real_val']]
if all(os.path.exists(f) for f in val_files):
    print(evaluate_weights(WEIGHTS, CFG).round(4).to_string())
else:
    print('данных для валидации нет (их готовит раздел 4 при TRAIN = True), пропускаю')

                score     acc  logloss     ece
synth_val      0.9723  0.9616   0.0965  0.0195
coco_val       0.9057  0.8575   0.2964  0.0147
ic15_val       0.9239  0.8886   0.2467  0.0353
textocr_val    0.8920  0.8366   0.3363  0.0135
hw_ru_val      0.9673  0.9612   0.1320  0.0291
plates_ru_val  0.9938  0.9933   0.0342  0.0261
real_all       0.9016  0.8517   0.3085  0.0114


### Сабмит

Делаем предсказания для всех 20 000 тестовых картинок и проверяем формат: две колонки `image_id` и `p_180`,
те же id и в том же порядке что в `sample_submission.csv`, вероятности от 0 до 1, без пропусков.

In [32]:
sub = make_submission(WEIGHTS, SUBMISSION, CFG['input']['width'])

# проверяем формат
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
assert list(sub.columns) == ['image_id', 'p_180']
assert sub.image_id.equals(sample.image_id)
assert sub.p_180.between(0, 1).all() and not sub.p_180.isna().any()
print('формат ок')
print(f'доля p_180 > 0.5: {(sub.p_180 > 0.5).mean():.3f}, неуверенных (0.1 < p < 0.9): {sub.p_180.between(0.1, 0.9).mean():.3f}')
sub.head()

сохранил submission.csv: 20000 строк, среднее p_180 = 0.5012
формат ок
доля p_180 > 0.5: 0.502, неуверенных (0.1 < p < 0.9): 0.192


,image_id,p_180
0,test_00000,0.075208
1,test_00001,0.955276
2,test_00002,0.999815
3,test_00003,0.012321
4,test_00004,0.995610
